# sanitise - PDF / image input: try it and measure it (Kaggle)

Reads PDFs (typed or scanned), photos and handwriting into text for Agents 1-3, and measures how
well it does on test documents whose text is known. Full description: `docs/document_input.md`.

**Before running:** right sidebar -> **Session options** -> Accelerator **GPU T4 x2**, **Internet
On** (needs a phone-verified Kaggle account). Run cell 1, then whichever of the others you need.

| Cell | What it does | Time (T4, rough) |
|---|---|---|
| 1 Setup | Clone the repo, install packages, Ollama and the vision model (Qwen3-VL 8B, ~6 GB) | ~5 min |
| 1b Real test set | Download the 150 real documents (50 health, 50 finance, 50 law) | ~3 min |
| 2 One document | Read one file (a test document, your own upload, or a URL) and print the text; optionally run Agents 1-3 on it | seconds (typed PDF) to ~1 min a scanned page |
| 3 Metrics | Score test documents (choose set, domain, number) and real handwritten lines | ~1 min a scanned page |
| 4 Look closer | One test document: the image, the gold text and what was read | seconds |
| 5 Bigger synthetic set | Make more synthetic documents (any number per type) | ~1 min |

**Test sets.** `data/ocr_test`: 45 synthetic documents with exact text (35 health, 5 finance,
5 law). `data/ocr_real`: 150 real documents, 50 per domain. Health: ClinOCR-Bench, clinical
documents printed, crumpled, copied and scanned (40), and real doctors' handwriting from
prescriptions (10). Finance: real scanned receipts (SROIE, 25) and receipt photos (CORD, 25). Law:
real SEC contracts, half PDF and half scanned (CUAD, 35), and real scanned forms (FUNSD, 15).

**The metrics.** CER, the character error rate: the share of characters wrong compared with the
known text (0% perfect, 2% = 2 wrong in 100). Word F1: words read correctly whatever their order.
Use it for receipts and forms, whose known text was rebuilt from word boxes and so has only an
approximate order. Also reported: private details read exactly (synthetic set), pages routed the
right way, header/footer removal.

**Your own file:** right sidebar -> **Input -> Upload** (it becomes a private Dataset under
`/kaggle/input/...`), then put its path in cell 2. Or paste a URL.

In [ ]:
# 1 Setup
BRANCH = "ingest"                 # the branch with the PDF / image input
OCR_MODEL = "qwen3-vl:8b-instruct"  # the vision model for scanned pages, photos and handwriting
# Cell 2 can also run Agents 1-3 on what it read: that needs the Gemini key (Kaggle secret
# GEMINI_API_KEY, Add-ons -> Secrets, attached to this notebook). False: reading only.
AGENTS = False
GEMINI_MODEL = "gemini-3.5-flash-lite,gemini-3.1-flash-lite"

import os
import subprocess
import sys
from pathlib import Path

WORK = Path("/kaggle/working")
os.environ["SANITISE_WORKDIR"] = "/tmp/sanitise"  # Ollama and the model, outside the saved output
REPO = WORK / "3rd_sem_el"
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH,
                    "https://github.com/raksha-sendhil/3rd_sem_el.git", str(REPO)], check=True)
os.chdir(REPO)
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU: Session options -> Accelerator -> GPU T4 x2"

!pip install -q -r requirements.txt
!pip install -q -e . --no-deps
sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO / "tools"))
for name in [m for m in sys.modules if m == "sanitise" or m.startswith(("sanitise.", "colab_cache"))]:
    del sys.modules[name]

import colab_cache

colab_cache.install_ollama_and_model(OCR_MODEL)
os.environ.update({"SANITISE_OCR_MODEL": OCR_MODEL, "SANITISE_LLM_BACKEND": "ollama"})
if AGENTS:
    from kaggle_secrets import UserSecretsClient
    os.environ.update({
        "SANITISE_AGENTS_BACKEND": "gemini",
        "SANITISE_AGENTS_MODEL": GEMINI_MODEL,
        "SANITISE_GEMINI_API_KEY": UserSecretsClient().get_secret("GEMINI_API_KEY"),
    })
print("ready: vision model", OCR_MODEL, "| agents", "Gemini" if AGENTS else "off")

In [ ]:
# 1b Real test set: 50 real documents per domain, downloaded from their public sources
PER_DOMAIN = 50                      # fewer = faster to download and to score
DOMAINS = "health,finance,law"

!python tools/fetch_real_testset.py --per-domain {PER_DOMAIN} --domains {DOMAINS}

In [ ]:
# 2 One document: read it and print the text (and, with SANITISE, Agents 1-3's rewrite)
# FILE: a test-set id (prescription_hw_01, health_clinocr_tables_t10_s2, finance_cord_03, ...),
# a path to your own file (e.g. /kaggle/input/my-docs/scan.pdf) or a URL.
FILE = "prescription_hw_01"
SANITISE = False   # also run Agents 1-3 on each part (needs AGENTS = True in cell 1)
STRIP = True       # remove repeated headers/footers after their first page
FORCE_VISION = False  # send typed PDF pages to the vision model too (to compare)

flags = " ".join(f for f, on in [("--sanitise", SANITISE), ("--no-strip", not STRIP),
                                  ("--force-vision", FORCE_VISION)] if on)
!python tools/ingest_file.py "{FILE}" {flags} --out /kaggle/working/last_read.txt

In [ ]:
# 3 Metrics (safe to re-run: documents already read are not read again)
SET = "data/ocr_test,data/ocr_real"  # synthetic, real, or both (comma-separated); cell 5's folder too
DOMAIN = None      # "health", "finance", "law" or None for all three
N_DOCS = 18        # how many documents; None = all. Picked one type at a time, so every type is covered
TYPES = ""         # only these types, comma-separated (e.g. "clinocr_handwriting,rx_handwritten_word")
IAM_LINES = 20     # real handwritten lines from the IAM database (0 = skip); internet needed
RUN = ""           # results folder name; "" = the model name
FRESH = False      # True: read everything again (after changing the prompt or the model)

from IPython.display import Markdown, display

out = f"/kaggle/working/ocr_results/{RUN or OCR_MODEL.replace(':', '_')}"
args = f'--set "{SET}" --iam {IAM_LINES} --out "{out}"'
if DOMAIN:
    args += f" --domain {DOMAIN}"
if N_DOCS is not None:
    args += f" --n {N_DOCS}"
if TYPES:
    args += f" --types {TYPES}"
if FRESH:
    args += " --fresh"
!python tools/ocr_eval.py {args} --quiet
display(Markdown(Path(out, "report.md").read_text()))
print("all numbers:", out + "/report.json", "| what was read:", out + "/pred/")

In [ ]:
# 4 Look closer at one test document: the image, the gold text and what was read
DOC = "health_clinocr_handwriting_t1_s2"   # any id from data/ocr_test or data/ocr_real

import html
import json

import pymupdf
from IPython.display import HTML, Image, display

gold = None
for folder in ["data/ocr_test", "data/ocr_real"]:
    gold_file = Path(folder, "gold.jsonl")
    if gold_file.exists():
        for line in gold_file.read_text().splitlines():
            g = json.loads(line)
            if g["id"] == DOC:
                gold, f = g, Path(folder, g["file"])
if gold is None:
    raise SystemExit(f"{DOC} is not in the test sets")
print(gold["about"], "|", gold.get("source", "synthetic"), "|", gold.get("licence", ""))
if f.suffix == ".pdf":
    for page in pymupdf.open(f):
        display(Image(data=page.get_pixmap(dpi=70).tobytes("png")))
else:
    display(Image(filename=str(f), width=600))
read = Path(out, "pred", f"{DOC}.txt")
read = read.read_text() if read.exists() else "(not read yet: run cell 3 with this document, or cell 2)"
box = "<td style='vertical-align:top;width:50%'><b>{}</b><pre style='white-space:pre-wrap'>{}</pre></td>"
display(HTML("<table><tr>" + box.format("Gold", html.escape(gold["gold"]))
             + box.format("Read", html.escape(read)) + "</tr></table>"))

In [ ]:
# 5 Optional: a bigger (or different) test set, then set SET in cell 3 to the folder printed
PER_TYPE = 10      # documents of each of the 9 types
SEED = 7           # another seed = other people, layouts, fonts and noise
TYPES_TO_MAKE = "" # "" = all types

SET_OUT = f"/kaggle/working/ocr_test_{PER_TYPE}x_seed{SEED}"
extra = f"--types {TYPES_TO_MAKE}" if TYPES_TO_MAKE else ""
!python tools/make_ocr_testset.py --per-type {PER_TYPE} --seed {SEED} --out "{SET_OUT}" {extra} | tail -n 1
print("now set SET =", repr(SET_OUT), "in cell 3")